In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/[Research] EEG/[Completed] Tesis/Baseline models/v2/Brain-computer-interfaces-master/notebooks

/content/drive/MyDrive/[Research] EEG/[Completed] Tesis/Baseline models/v2/Brain-computer-interfaces-master/notebooks


In [3]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [4]:
# -*- coding: utf-8 -*-
"""
12-Class SSVEP EEG Dataset - Classification Using Convolutional Neural Network
User-Dependent Training using Complex Spectrum Features (10-Fold Cross-validation)
Following implementation is an asynchronous SSVEP BCI
using Convolutional Neural Network classification for 1 second data length
"""
import numpy as np
import scipy.io as sio
from sklearn.model_selection import KFold

from keras.utils import to_categorical
from keras import optimizers
from keras.losses import categorical_crossentropy

from scripts import ssvep_utils as su


In [5]:
CNN_PARAMS = {
    'batch_size': 64,
    'epochs': 50,
    'droprate': 0.25,
    'learning_rate': 0.001,
    'lr_decay': 0.0,
    'l2_lambda': 0.0001,
    'momentum': 0.9,
    'kernel_f': 10,
    'n_ch': 3,
    'num_classes': 4}

FFT_PARAMS = {
    'resolution': 0.5,
    'start_frequency': 3.0,
    'end_frequency': 35.0,
    'sampling_rate': 250
}

In [17]:
all_acc = np.zeros((10, 1))
window_length = 0.1

for subject in range(0, 10):
  all_subjects_train_X = []
  all_subjects_train_Y = []

  for non_target_sub in range(0,10):
    if subject == non_target_sub:
      continue

    dataset = sio.loadmat(f'/content/drive/MyDrive/[Research] EEG/[Completed] Tesis/Datasets/Noisy-UTEC/Noisy-UTEC mat files/S{non_target_sub+1}.mat')
    eeg = np.array(dataset['eeg'], dtype='float32')
    eeg = np.swapaxes(eeg,1,2)
    eeg = np.swapaxes(eeg,2,3)

    CNN_PARAMS['num_classes'] = eeg.shape[0]
    CNN_PARAMS['n_ch'] = eeg.shape[1]
    total_trial_len = eeg.shape[2]
    num_trials = eeg.shape[3]
    sample_rate = 256

    filtered_data = su.get_filtered_eeg(eeg, 4, 30, 4, sample_rate,
                                        onset = 250,
                                        delay = 0.135,
                                        signal_end = 3.5)
    eeg = []

    window_len = window_length
    shift_len = window_length

    segmented_data = su.get_segmented_epochs(filtered_data, window_len, shift_len, sample_rate)
    segmented_data = np.expand_dims(segmented_data[:,:,:,0,:],axis = 3)
    filtered_data = []

    features_data = su.complex_spectrum_features(segmented_data, FFT_PARAMS)
    segmented_data = []

    #Combining the features into a matrix of dim [features X channels X classes X trials*segments]
    features_data = np.reshape(features_data, (features_data.shape[0], features_data.shape[1],
                                              features_data.shape[2], features_data.shape[3]*features_data.shape[4]))

    train_data = features_data[:, :, 0, :].T
    #Reshaping the data into dim [classes*trials*segments X channels X features]
    for target in range(1, features_data.shape[2]):
        train_data = np.vstack([train_data, np.squeeze(features_data[:, :, target, :]).T])

    #Finally reshaping the data into dim [classes*trials*segments X channels X features X 1]
    train_data = np.reshape(train_data, (train_data.shape[0], train_data.shape[1], train_data.shape[2], 1))

    total_epochs_per_class = features_data.shape[3]
    features_data = []

    class_labels = np.arange(CNN_PARAMS['num_classes'])
    labels = (np.tile(class_labels, (total_epochs_per_class, 1)).T).ravel()
    labels = to_categorical(labels)

    # Merging subject train part
    all_subjects_train_X.append(train_data)
    all_subjects_train_Y.append(labels)

  all_subjects_train_X = np.array(all_subjects_train_X)
  new_shape = (all_subjects_train_X.shape[0]*all_subjects_train_X.shape[1],all_subjects_train_X.shape[2],all_subjects_train_X.shape[3],1)
  all_subjects_train_X = all_subjects_train_X.reshape(new_shape)

  all_subjects_train_Y = np.array(all_subjects_train_Y)
  new_shape = (all_subjects_train_Y.shape[0]*all_subjects_train_Y.shape[1],all_subjects_train_Y.shape[2])
  all_subjects_train_Y = np.reshape(all_subjects_train_Y, new_shape)

  ## Test part
  dataset = sio.loadmat(f'/content/drive/MyDrive/[Research] EEG/[Completed] Tesis/Datasets/Noisy-UTEC/Noisy-UTEC mat files/S{subject+1}.mat')
  eeg = np.array(dataset['eeg'], dtype='float32')
  eeg = np.swapaxes(eeg,1,2)
  eeg = np.swapaxes(eeg,2,3)

  CNN_PARAMS['num_classes'] = eeg.shape[0]
  CNN_PARAMS['n_ch'] = eeg.shape[1]
  total_trial_len = eeg.shape[2]
  num_trials = eeg.shape[3]
  sample_rate = 256

  filtered_data = su.get_filtered_eeg(eeg, 4, 30, 4, sample_rate,
                                        onset = 250,
                                        delay = 0.135,
                                        signal_end = 3.5)
  eeg = []

  window_len = window_length
  shift_len = window_length

  segmented_data = su.get_segmented_epochs(filtered_data, window_len, shift_len, sample_rate)
  segmented_data = np.expand_dims(segmented_data[:,:,:,0,:],axis = 3)
  filtered_data = []

  features_data = su.complex_spectrum_features(segmented_data, FFT_PARAMS)
  segmented_data = []

  #Combining the features into a matrix of dim [features X channels X classes X trials*segments]
  features_data = np.reshape(features_data, (features_data.shape[0], features_data.shape[1],
                                            features_data.shape[2], features_data.shape[3]*features_data.shape[4]))

  train_data = features_data[:, :, 0, :].T
  #Reshaping the data into dim [classes*trials*segments X channels X features]
  for target in range(1, features_data.shape[2]):
      train_data = np.vstack([train_data, np.squeeze(features_data[:, :, target, :]).T])

  #Finally reshaping the data into dim [classes*trials*segments X channels X features X 1]
  all_subjects_test_X = np.reshape(train_data, (train_data.shape[0], train_data.shape[1], train_data.shape[2]))

  total_epochs_per_class = features_data.shape[3]
  features_data = []

  class_labels = np.arange(CNN_PARAMS['num_classes'])
  labels = (np.tile(class_labels, (total_epochs_per_class, 1)).T).ravel()
  all_subjects_test_Y = to_categorical(labels)

  x_tr, x_ts = all_subjects_train_X, all_subjects_test_X
  y_tr, y_ts = all_subjects_train_Y, all_subjects_test_Y
  input_shape = np.array([x_tr.shape[1], x_tr.shape[2], x_tr.shape[3]])

  fold = 0
  # print("Subject:", subject+1, "Fold:", fold+1, "Training...")

  model = su.CNN_model(input_shape, CNN_PARAMS)

  sgd = optimizers.SGD(learning_rate=CNN_PARAMS['learning_rate'], decay=CNN_PARAMS['lr_decay'],
                        momentum=CNN_PARAMS['momentum'], nesterov=False)
  model.compile(loss=categorical_crossentropy, optimizer=sgd, metrics=["accuracy"])
  history = model.fit(x_tr, y_tr, batch_size=CNN_PARAMS['batch_size'],
                      epochs=CNN_PARAMS['epochs'], verbose=0)

  score = model.evaluate(x_ts, y_ts, verbose=0)
  # print("%s: %.2f%%" % (model.metrics_names[1], score[1]*100))
  print(score[1]*100)

  # print("...................................................")
  # print(all_acc[subject])
  # print("...................................................")

13.333334028720856
21.666666865348816
36.666667461395264
10.000000149011612
18.333333730697632
25.0
23.333333432674408
28.333333134651184
33.33333432674408
20.000000298023224
